# Bước 2 - BM25 Okapi Retrieval & Đánh giá Truy xuất Bằng chứng
## Module 7.2: Information Retrieval (BM25 vs TF-IDF Baseline)

---

### Mục tiêu:
Dựa vào Tuyên bố (Claim), tự động tìm kiếm Top-$K$ câu bằng chứng trong kho câu của bài báo tương ứng.
- Đánh giá Recall@1, Recall@3, Recall@5, MRR đối chứng giữa **Okapi BM25** và **TF-IDF**.
- Xuất danh sách Top-5 câu ứng viên cho mỗi Claim vào file `outputs/bm25_results.csv`.

In [1]:
# ======================================================================
# 1. KHAI BÁO CÁC THƯ VIỆN CẦN THIẾT
# ======================================================================
import os  # Thư viện tương tác hệ điều hành (thư mục, đường dẫn)
import re  # Thư viện Regular Expressions để tách từ và làm sạch chuỗi
import time  # Thư viện đo thời gian thực thi thuật toán
from pathlib import Path  # Xử lý đường dẫn file/folder theo hướng đối tượng

import matplotlib.pyplot as plt  # Thư viện vẽ đồ thị, biểu đồ
import numpy as np  # Thư viện tính toán mảng và ma trận số học
import pandas as pd  # Thư viện bảng dữ liệu (DataFrame)
from rank_bm25 import BM25Okapi  # Thuật toán xếp hạng Okapi BM25 chuẩn
from sklearn.feature_extraction.text import TfidfVectorizer  # Bộ vector hóa TF-IDF
from sklearn.metrics.pairwise import cosine_similarity  # Hàm tính độ tương đồng Cosine

# ======================================================================
# 2. TỰ ĐỘNG ĐỊNH VỊ THƯ MỤC GỐC DỰ ÁN (PROJECT_ROOT)
# ======================================================================
current_dir = Path.cwd().resolve()
candidates = [current_dir, current_dir.parent, current_dir.parent.parent]
PROJECT_ROOT = current_dir
for cand in candidates:
    if (cand / 'data').exists() and (cand / 'notebooks').exists():
        PROJECT_ROOT = cand
        break

# Đường dẫn file tập dữ liệu Dev đã làm sạch chung
DEV_PATH = PROJECT_ROOT / 'data/processed/common_cleaned/vifactcheck_dev_common_cleaned.csv'
# Thư mục lưu các kết quả truy xuất bằng chứng
OUTPUT_DIR = PROJECT_ROOT / 'data/processed/retrieval'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)  # Tạo thư mục nếu chưa có
# File chứa kho tất cả các câu ứng viên (đã tách từ bài báo ở Bước 1)
CAND_PATH = OUTPUT_DIR / 'evidence_candidates.csv'

# Nạp dữ liệu vào DataFrame
df_dev = pd.read_csv(DEV_PATH)
df_candidates = pd.read_csv(CAND_PATH)
print(f'✓ Nạp thành công {len(df_dev):,} claims và {len(df_candidates):,} candidates.')


✓ Nạp thành công 723 claims và 12,823 candidates.


In [2]:
# ======================================================================
# 1. HÀM TÁCH TỪ CHO HỆ THỐNG TRUY XUẤT (TOKENIZATION)
# ======================================================================
def tokenize_for_ir(text: str) -> list[str]:
    """Tách câu thành danh sách các từ đơn giản, bỏ qua dấu câu."""
    if not isinstance(text, str):  # Nếu không phải chuỗi văn bản (NaN, None)
        return []  # Trả về danh sách rỗng
    # Chuyển về chữ thường và dùng regex \b\w+\b để lấy các từ nguyên vẹn
    return re.findall(r'\b\w+\b', text.lower())

# ======================================================================
# 2. LỚP BỘ TRUY XUẤT BM25 (BM25 RETRIEVER)
# ======================================================================
class BM25Retriever:
    """Lớp phụ trách tìm kiếm câu liên quan nhất trong bài báo bằng Okapi BM25."""
    def __init__(self, candidates: list[str]):
        self.candidates = candidates  # Danh sách các câu gốc trong bài báo
        # Tách từ cho từng câu để tạo thành tập ngữ liệu (corpus) cho BM25
        self.tokenized_corpus = [tokenize_for_ir(doc) for doc in candidates]
        # Khởi tạo mô hình toán học BM25Okapi trên kho câu đã tách từ
        self.bm25 = BM25Okapi(self.tokenized_corpus)
        
    def retrieve(self, query: str, top_k: int = 5) -> list[tuple[str, float, int]]:
        """Nhận câu Tuyên bố, trả về Top-K câu có điểm BM25 cao nhất."""
        tokenized_query = tokenize_for_ir(query)  # Tách từ câu truy vấn
        if not tokenized_query:  # Nếu câu truy vấn không có từ nào
            return []
        # Tính điểm BM25 của câu truy vấn so với tất cả các câu trong bài báo
        scores = self.bm25.get_scores(tokenized_query)
        # Sắp xếp chỉ số theo điểm giảm dần [::-1] và cắt lấy K câu đầu tiên
        top_indices = np.argsort(scores)[::-1][:top_k]
        # Trả về danh sách bộ 3: (nội dung câu, điểm số BM25, chỉ số câu)
        return [(self.candidates[idx], float(scores[idx]), int(idx)) for idx in top_indices]

# ======================================================================
# 3. CÁC HÀM XÁC THỰC CÂU BẰNG CHỨNG VÀNG (GOLD EVIDENCE CHECK)
# ======================================================================
def normalize_evidence_text(t: str) -> str:
    """Làm sạch khoảng trắng thừa và chuyển về chữ thường để so sánh văn bản."""
    return re.sub(r'\s+', ' ', str(t).lower()).strip()

def is_evidence_match(candidate: str, gold_evidence: str, min_overlap: float = 0.6) -> bool:
    """Kiểm tra câu do máy tìm ra có trùng khớp với Bằng chứng Vàng thật sự không."""
    if not isinstance(gold_evidence, str) or not gold_evidence.strip():
        return False  # Không có câu vàng thì trả về False
    cand_norm = normalize_evidence_text(candidate)
    gold_norm = normalize_evidence_text(gold_evidence)
    # Trường hợp 1: Một trong 2 câu là chuỗi con của nhau (khớp hoàn hảo)
    if cand_norm in gold_norm or gold_norm in cand_norm:
        return True
    # Trường hợp 2: So sánh tập từ vựng giao nhau
    c_words = set(cand_norm.split())
    g_words = set(gold_norm.split())
    if not c_words or not g_words:
        return False
    # Nếu số từ trùng lặp >= 60% so với độ dài câu ngắn hơn thì xem như tìm đúng
    return len(c_words.intersection(g_words)) / min(len(c_words), len(g_words)) >= min_overlap

print('✓ Bộ máy BM25 và hàm kiểm thử trùng khớp Bằng chứng Vàng đã sẵn sàng.')


✓ Bộ máy BM25 và hàm kiểm thử trùng khớp Bằng chứng Vàng đã sẵn sàng.


In [3]:
# ======================================================================
# 1. TIẾN HÀNH TRUY XUẤT CHO TẤT CẢ CÁC CLAIMS TRONG TẬP DEV
# ======================================================================
TOP_K = 5  # Số lượng câu bằng chứng cần lấy cho mỗi tuyên bố
retrieved_rows = []  # Danh sách gom toàn bộ kết quả của các bài báo

# Duyệt qua từng dòng trong tập dữ liệu kiểm định Dev
for _, row in df_dev.iterrows():
    claim_id = f"dev_{row['index']}"  # Mã định danh cho từng tuyên bố
    claim_text = str(row['Statement'])  # Nội dung câu tuyên bố
    # Lấy nội dung câu bằng chứng vàng nếu có
    gold_text = str(row['Evidence']) if pd.notna(row['Evidence']) else ''
    
    # Lọc danh sách tất cả các câu thuộc về đúng bài báo tương ứng của claim này
    article_candidates = df_candidates[df_candidates['claim_id'] == claim_id]['sentence_text'].tolist()
    if not article_candidates:  # Nếu bài báo không có câu nào thì bỏ qua
        continue
        
    # Khởi tạo mô hình BM25 cho bài báo này
    retriever = BM25Retriever(article_candidates)
    # Tìm ra 5 câu có điểm liên quan cao nhất
    top_results = retriever.retrieve(claim_text, top_k=TOP_K)
    
    # Duyệt qua 5 câu kết quả, đánh số thứ tự xếp hạng từ 1 đến 5
    for rank, (cand_sent, score, _) in enumerate(top_results, 1):
        # Nếu không phải nhãn NEI (labels != 2), kiểm tra xem câu này có đúng là Bằng chứng Vàng không
        is_hit = is_evidence_match(cand_sent, gold_text) if row['labels'] != 2 else False
        retrieved_rows.append({
            'claim_id': claim_id,  # ID câu tuyên bố
            'claim_index': row['index'],  # Chỉ số index gốc
            'claim': claim_text,  # Câu tuyên bố
            'retrieved_evidence': cand_sent,  # Câu bằng chứng do BM25 tìm được
            'bm25_score': round(score, 4),  # Điểm số liên quan của BM25
            'rank': rank,  # Vị trí thứ hạng (Rank 1, 2, 3, 4, 5)
            'is_gold': is_hit,  # Cờ True/False: Có phải Bằng chứng Vàng không
            'label': row['labels'],  # Nhãn gốc (0: Supported, 1: Refuted, 2: NEI)
            'gold_evidence': gold_text if row['labels'] != 2 else ''
        })

# Gom kết quả thành DataFrame
df_retrieved = pd.DataFrame(retrieved_rows)
output_bm25_path = OUTPUT_DIR / 'bm25_results.csv'
# Lưu bảng kết quả xuống đĩa
df_retrieved.to_csv(output_bm25_path, index=False)

# ======================================================================
# 2. ĐÁNH GIÁ CHỈ SỐ HIỆU NĂNG (RECALL@K & MRR) TRÊN TẬP NON-NEI
# ======================================================================
# Chỉ lọc các mẫu SUPPORTED và REFUTED (nhãn != 2) vì chỉ chúng mới có Bằng chứng Vàng
eval_df = df_retrieved[df_retrieved['label'] != 2]
# Tổng số claim thực tế cần đánh giá
n_claims = eval_df['claim_id'].nunique()

# Recall@1: Tỷ lệ claim có Bằng chứng Vàng nằm ngay vị trí số 1
r1 = eval_df[(eval_df['rank'] == 1) & (eval_df['is_gold'] == True)]['claim_id'].nunique() / n_claims
# Recall@3: Tỷ lệ claim có Bằng chứng Vàng nằm trong Top 3 câu đầu
r3 = eval_df[(eval_df['rank'] <= 3) & (eval_df['is_gold'] == True)]['claim_id'].nunique() / n_claims
# Recall@5: Tỷ lệ claim có Bằng chứng Vàng nằm trong Top 5 câu đầu
r5 = eval_df[(eval_df['rank'] <= 5) & (eval_df['is_gold'] == True)]['claim_id'].nunique() / n_claims

# Lọc ra các dòng khớp đúng Bằng chứng Vàng để tính thứ hạng trung bình MRR
gold_rows = eval_df[eval_df['is_gold'] == True]
# Lấy thứ hạng nhỏ nhất (vị trí đầu tiên) mà câu vàng xuất hiện, tính trung bình 1 / rank
mrr = (1.0 / gold_rows.groupby('claim_id')['rank'].min()).sum() / n_claims

# In báo cáo kết quả hoàn chỉnh
print(f'✓ Đã lưu bảng kết quả BM25 ({len(df_retrieved):,} dòng) tại: {output_bm25_path.name}')
print('\n' + '=' * 60)
print('KẾT QUẢ TRUY XUẤT BM25 (Okapi):')
print(f'• Recall@1: {r1*100:.2f}%')
print(f'• Recall@3: {r3*100:.2f}%')
print(f'• Recall@5: {r5*100:.2f}%')
print(f'• MRR:      {mrr:.4f}')
print('=' * 60)
display(df_retrieved.head(6))


✓ Đã lưu bảng kết quả BM25 (3,608 dòng) tại: bm25_results.csv

KẾT QUẢ TRUY XUẤT BM25 (Okapi):
• Recall@1: 89.80%
• Recall@3: 96.40%
• Recall@5: 97.00%
• MRR:      0.9297


,claim_id,claim_index,claim,retrieved_evidence,bm25_score,rank,is_gold,label,gold_evidence
0,dev_6040,6040,"Vào tháng 4.1930 TL Nhà vua Na Uy Harald V, Vu...",Vua hề Charlie Chaplin (vua hề Sác lô) và vợ t...,56.3285,1,True,1,Vua hề Charlie Chaplin (vua hề Sác lô) và vợ t...
1,dev_6040,6040,"Vào tháng 4.1930 TL Nhà vua Na Uy Harald V, Vu...","Trong số đó, có vua hề Charlie Chaplin (vua hề...",51.3780,2,True,1,Vua hề Charlie Chaplin (vua hề Sác lô) và vợ t...
2,dev_6040,6040,"Vào tháng 4.1930 TL Nhà vua Na Uy Harald V, Vu...",Swore Oath ở khách sạn Saigon Morin năm 2004 T...,7.4638,3,False,1,Vua hề Charlie Chaplin (vua hề Sác lô) và vợ t...
3,dev_6040,6040,"Vào tháng 4.1930 TL Nhà vua Na Uy Harald V, Vu...","Phu nhân cựu Tổng thống Pháp, bà Bernadette Ch...",6.8516,4,False,1,Vua hề Charlie Chaplin (vua hề Sác lô) và vợ t...
4,dev_6040,6040,"Vào tháng 4.1930 TL Nhà vua Na Uy Harald V, Vu...","Saigon Morin, khách sạn 4 sao hàng đầu tại Huế...",5.2987,5,False,1,Vua hề Charlie Chaplin (vua hề Sác lô) và vợ t...
5,dev_5599,5599,Nhiều chi bộ chỉ mua báo đảng mà không quan tâ...,"Nhiều chi bộ mới chỉ dừng ở việc mua báo đảng,...",44.5533,1,True,1,"Việc mua, đọc, sử dụng báo, tạp chí của Đảng t..."
